In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types
from pyspark.sql import functions as F
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

In [ ]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [ ]:
!wget https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz

In [ ]:
# using bash to convert the fhvhv_tripdata from .gz to .csv
!gzip -dc fhvhv_tripdata_2021-01.csv.gz > fhvhv_tripdata_2021-01.csv

In [ ]:
# verifying row count and dataformate using bash
!wc -l fhvhv_tripdata_2021-01.csv

In [ ]:
# Checking my directory is in the git repo for version control
!pwd


In [ ]:
# running spark command to read the .csv in the current directory
df = spark.read \
    .option("header", "true") \
    .csv('fhvhv_tripdata_2021-01.csv')

In [ ]:
# Checkng the schema of the dataframe read
df.schema

In [ ]:
# using bash to get the first 100 rows in the datafram
!head -n 1001 fhvhv_tripdata_2021-01.csv > head.csv

In [ ]:
!head -n 10 head.csv

In [ ]:
import pandas as pd

In [ ]:
df_pandas = pd.read_csv('head.csv')

In [ ]:
df_pandas.dtypes

In [ ]:
# using spark to see the schema
spark.createDataFrame(df_pandas).schema

In [ ]:
# Creating the schema with the correct datatype
schema = types.StructType(
    [
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.DoubleType(), True)
    ]
)

In [ ]:
# importing the entire data with the correct schema
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv('fhvhv_tripdata_2021-01.csv')

In [ ]:
df.head(5)

In [ ]:
# repartitioning a file in spark, it take a dataframe and partition the datafram
df.repartition(24)

In [ ]:
df.write.parquet('fhvhv/2021/01/')

### `Spark Dataframes`
* Transformation  are lazy (not executed immediately)
* Tranformation examplses are select, filter, change datatime
* Actions -Eager(trigger execution immediately)
* Example of action include Show, Take, head, write

In [ ]:
df = spark.read.parquet('fhvhv/2021/01/')

In [ ]:
# checking the schema of the data
df.printSchema()

In [ ]:
df.select('dropoff_datetime').show()

In [ ]:
from pyspark.sql import functions as F

df \
    .withColumn('pickup_datetime', F.to_date(F.col('pickup_datetime'))) \
    .withColumn('dropoff_datetime', F.to_date(F.col('dropoff_datetime'))) \
    .select('pickup_datetime','dropoff_datetime','PULocationID','DOLocationID') \
    .show()

In [ ]:
df.select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
  .filter(df.hvfhs_license_num == 'HV0003')

In [ ]:
def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    elif num % 3 == 0:
        return f'a/{num:03x}'
    else:
        return f'e/{num:03x}'

In [ ]:
crazy_stuff('B02884')

In [ ]:
crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())

In [ ]:
###### preparing data for assignment